## Con el video de la camara detectar una imagen roja.

Una secuencia típica es:

- cargar la imagen
- aplicar suavizado (gaussiano)
- eligir objectos con color rojo
- aplicar erosión
- aplicar dilatación
- aplicar Canny
- mostrar resultados


In [4]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

# Guardar salida opcional
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('video_rojo_1ventana.avi', fourcc, 20.0, (640, 480))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    
    blurred = cv2.GaussianBlur(frame, (7, 7), 0)

    # 1) Filtrar rojo en HSV
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

    lower_red1 = np.array([0, 100, 100], dtype=np.uint8)
    upper_red1 = np.array([10, 255, 255], dtype=np.uint8)

    lower_red2 = np.array([160, 100, 100], dtype=np.uint8)
    upper_red2 = np.array([180, 255, 255], dtype=np.uint8)

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = cv2.bitwise_or(mask1, mask2)

    # 2) Limpieza de la máscara
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # 3) Aplicar erosión y dilatación solo sobre la parte roja
    kernel_erode = np.ones((3, 3), np.uint8)
    red_mask = cv2.erode(red_mask, kernel_erode, iterations=1)

    kernel_dilate = np.ones((5, 5), np.uint8)
    red_mask = cv2.dilate(red_mask, kernel_dilate, iterations=1)

    # 4) Canny solo sobre la zona roja
    edges = cv2.Canny(red_mask, 50, 150)

    # 5) Mostrar todo en una sola ventana
    result = blurred.copy()
    result[red_mask > 0] = (0, 0, 255)   # rojo en BGR
    result[edges > 0] = (0, 255, 255)   # contorno amarillo

    # Guardar video
    out.write(result)

    cv2.imshow("Original", frame)
    cv2.imshow("Video con rojo", result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

## INPUT de los profe
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
 
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 25))


import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
 
 
stop_button = widgets.ToggleButton(
    value=False,
    description='Detener Cámara',
    button_style='danger',
    icon='stop'
)
display(stop_button)
 
image_widget = widgets.Image(format='jpeg', width=960)
display(image_widget)
 
cap = cv2.VideoCapture(0)
 
lower_green = np.array([35, 45, 40])
upper_green = np.array([85, 255, 255])
 
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
 
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 25))
 
EDGE_COLOR = [255, 255, 0]
 
try:
    while not stop_button.value:
        ret, frame = cap.read()
        if not ret:
            break
 
        blurred = cv2.GaussianBlur(frame, (7, 7), 0)
        hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
   
        mask = cv2.inRange(hsv, lower_green, upper_green)
 
              mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open, iterations=2)
       
        mask_solid = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel_close, iterations=2)
       
        frasco_aislado = cv2.bitwise_and(frame, frame, mask=mask_solid)
      
        edges = cv2.Canny(mask_solid, 50, 150)
        edges_dilated = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
 
        resultado = frasco_aislado.copy()
        resultado[edges_dilated > 0] = EDGE_COLOR
 
        vista_original = frame.copy()
        cv2.putText(vista_original, "Original", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(resultado, "Morfologia + Canny Puro", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2, cv2.LINE_AA)
 
        comparativa = np.hstack([vista_original, resultado])
     
        _, jpeg = cv2.imencode('.jpg', comparativa)
        image_widget.value = jpeg.tobytes()
 
finally:
    cap.release()
    print("Cámara liberada correctamente.")
 

In [18]:
pip install ipywidgets


  Using cached ipywidgets-8.1.9-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.16-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.9-py3-none-any.whl (140 kB)
Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl (217 kB)
Using cached widgetsnbextension-4.0.16-py3-none-any.whl (2.2 MB)

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [19]:
## INPUT de los profe
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
 
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 25))

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
 
 
stop_button = widgets.ToggleButton(
    value=False,
    description='Detener Cámara',
    button_style='danger',
    icon='stop'
)
display(stop_button)
 
image_widget = widgets.Image(format='jpeg', width=960)
display(image_widget)
 
cap = cv2.VideoCapture(0)
 
lower_green = np.array([35, 45, 40])
upper_green = np.array([85, 255, 255])
 
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
 
kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 25))
 
EDGE_COLOR = [255, 255, 0]
 
try:
    while not stop_button.value:
        ret, frame = cap.read()
        if not ret:
            break
 
        blurred = cv2.GaussianBlur(frame, (7, 7), 0)
        hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
   
        mask = cv2.inRange(hsv, lower_green, upper_green)
 
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open, iterations=2)
       
        mask_solid = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel_close, iterations=2)
       
        frasco_aislado = cv2.bitwise_and(frame, frame, mask=mask_solid)
      
        edges = cv2.Canny(mask_solid, 50, 150)
        edges_dilated = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
 
        resultado = frasco_aislado.copy()
        resultado[edges_dilated > 0] = EDGE_COLOR
 
        vista_original = frame.copy()
        cv2.putText(vista_original, "Original", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(resultado, "Morfologia + Canny Puro", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2, cv2.LINE_AA)
 
        comparativa = np.hstack([vista_original, resultado])
     
        _, jpeg = cv2.imencode('.jpg', comparativa)
        image_widget.value = jpeg.tobytes()
 
finally:
    cap.release()
    print("Cámara liberada correctamente.")
 

ToggleButton(value=False, button_style='danger', description='Detener Cámara', icon='stop')

Image(value=b'', format='jpeg', width='960')

Cámara liberada correctamente.


KeyboardInterrupt: 